# RoBERTa-base · SQuAD v2 — Pipeline completo (EDA → Training → Error Analysis)

Este notebook hace **todo de una sola vez**. Con **Entorno de ejecución → Ejecutar todas** genera
la carpeta `logs_roberta/` completa en Google Drive (`MyDrive/roberta_squad2/logs_roberta/`).

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU**.

| Parte | Qué hace | Archivos en `logs_roberta/` |
|---|---|---|
| A · EDA | Estadísticas y gráficas de SQuAD v2 | `eda_squad2.png`, `eda_stats.json` |
| B · Training | Fine-tuning de `FacebookAI/roberta-base`, evaluación, latencia CPU | `loss.png`, `train_results.json`, `eval_results.json` |
| C · Error Analysis | Threshold, errores, README y comparación | `threshold_search.png`, `error_analysis.json`, `README.md`, `ROBERTA_COMPARISON.md` |

Si Colab se desconecta durante el entrenamiento, vuelve a **Ejecutar todas**: la Parte B continúa
desde el último checkpoint guardado en Drive (la Parte A se repite, tarda pocos minutos).

Es el mismo código que los tres notebooks separados (`01_…`, `02_…`, `03_…`), unido en uno.

## Instalación y rutas

In [ ]:
# ⚠️ Es normal que pip muestre un aviso de 'dependency conflicts' con gradio/diffusers:
#    este notebook no usa esos paquetes. Tras esta celda, la primera vez:
#    Entorno de ejecución → Reiniciar sesión, y luego Ejecutar todas de nuevo.
# Colab ya trae torch, numpy y matplotlib. Fijamos transformers < 5 para que la API
# (Trainer, pipeline) sea la misma con la que se escribió este notebook.
%pip install -q "transformers>=4.46,<5" "datasets>=3.0" "evaluate>=0.4" "accelerate>=0.34"

In [ ]:
import os, sys

# ── Rutas ─────────────────────────────────────────────────────────────
# En Colab todo se guarda en Google Drive para que no se pierda si la sesión
# se desconecta. Fuera de Colab se usa la raíz del repositorio.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/roberta_squad2"
else:
    BASE_DIR = os.path.abspath("..")   # el notebook vive en pipeline_roberta/

LOGS_DIR   = os.path.join(BASE_DIR, "logs_roberta")                 # resultados para GitHub
MODEL_DIR  = os.path.join(BASE_DIR, "roberta-base-squad2-final")    # modelo final (NO va a GitHub)
CKPT_DIR   = os.path.join(BASE_DIR, "checkpoints")                  # checkpoints intermedios
PRED_FILE  = os.path.join(BASE_DIR, "predictions_val.json")         # predicciones de validación
os.makedirs(LOGS_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("LOGS_DIR:", LOGS_DIR)

---
# Parte A · EDA


**Objetivo:** entender el dataset antes de entrenar.

Este notebook:
1. Descarga SQuAD v2 (`rajpurkar/squad_v2`).
2. Cuenta preguntas **contestables** y **no contestables** en train y validation.
3. Mide la longitud de contextos, preguntas y respuestas.
4. Mide la longitud de los contextos **en tokens de RoBERTa** y cuántas ventanas
   (`max_seq_length=384`, `doc_stride=128`) necesita cada ejemplo.
5. Guarda en `logs_roberta/`:
   - `eda_squad2.png` — gráficas
   - `eda_stats.json` — estadísticas

No necesita GPU. Tiempo aproximado: 3–5 minutos.

## 2. Cargar el dataset

In [ ]:
import json, collections
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "FacebookAI/roberta-base"
MAX_SEQ_LENGTH   = 384
DOC_STRIDE       = 128

ds = load_dataset("rajpurkar/squad_v2")
train, val = ds["train"], ds["validation"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(ds)

In [ ]:
# Un ejemplo contestable y uno no contestable
def show(ex):
    print("Pregunta :", ex["question"])
    print("Contexto :", ex["context"][:300], "...")
    print("Respuesta:", ex["answers"]["text"] if ex["answers"]["text"] else "(sin respuesta)")
    print("-" * 80)

show(next(ex for ex in val if ex["answers"]["text"]))
show(next(ex for ex in val if not ex["answers"]["text"]))

## 3. Contestables vs. no contestables

En SQuAD v2 una pregunta **no contestable** tiene la lista `answers["text"]` vacía.
El modelo debe aprender a devolver una respuesta vacía en esos casos.

In [ ]:
def is_unanswerable(ex):
    return len(ex["answers"]["text"]) == 0

train_unans = [is_unanswerable(ex) for ex in train]
val_unans   = [is_unanswerable(ex) for ex in val]

for name, split, flags in [("train", train, train_unans), ("validation", val, val_unans)]:
    n, u = len(split), sum(flags)
    print(f"{name:<11} total={n:>7,}  contestables={n-u:>7,} ({(n-u)/n:.1%})  no contestables={u:>6,} ({u/n:.1%})")

## 4. Longitudes: contexto, pregunta y respuesta

In [ ]:
# Contexto (caracteres) — por ejemplo, como en el EDA de ModernBERT
ctx_chars = np.array([len(c) for c in train["context"]])

# Contextos únicos en tokens de RoBERTa (muchas preguntas comparten contexto)
unique_ctx = list(dict.fromkeys(train["context"]))
ctx_tokens = np.array([len(ids) for ids in tokenizer(unique_ctx, add_special_tokens=False)["input_ids"]])

# Preguntas (palabras)
q_words = np.array([len(q.split()) for q in train["question"]])
q_words_ans   = q_words[~np.array(train_unans)]
q_words_unans = q_words[np.array(train_unans)]

# Respuestas (todas las respuestas de referencia del train)
answers_flat = [a for ex in train["answers"] for a in ex["text"]]
ans_words = np.array([len(a.split()) for a in answers_flat])
ans_chars = np.array([len(a) for a in answers_flat])

print(f"Contextos únicos: {len(unique_ctx):,}  |  artículos: {len(set(train['title'])):,}")
print(f"Contexto (tokens) media={ctx_tokens.mean():.0f} mediana={np.median(ctx_tokens):.0f} "
      f"p95={np.percentile(ctx_tokens,95):.0f} max={ctx_tokens.max()}")
print(f"Contextos con más de {MAX_SEQ_LENGTH} tokens: {(ctx_tokens > MAX_SEQ_LENGTH).mean():.1%}")
print(f"Pregunta (palabras) media={q_words.mean():.1f}")
print(f"Respuesta (palabras) media={ans_words.mean():.1f} mediana={np.median(ans_words):.0f} max={ans_words.max()}")

## 5. Ventanas por ejemplo (sliding window)

Si pregunta + contexto no caben en 384 tokens, el contexto se divide en varias
ventanas que se solapan 128 tokens. Aquí contamos cuántas ventanas (*features*)
genera cada split: es el número real de muestras que verá el modelo.

In [ ]:
def count_windows(split, chunk=5000):
    counts = []
    for i in range(0, len(split), chunk):
        part = split[i:i + chunk]
        enc = tokenizer([q.strip() for q in part["question"]], part["context"],
                        max_length=MAX_SEQ_LENGTH, truncation="only_second",
                        stride=DOC_STRIDE, return_overflowing_tokens=True)
        counts.append(np.bincount(enc["overflow_to_sample_mapping"], minlength=len(part["id"])))
    return np.concatenate(counts)

train_windows = count_windows(train)
val_windows   = count_windows(val)
for name, w in [("train", train_windows), ("validation", val_windows)]:
    print(f"{name:<11} features={w.sum():>7,}  ejemplos con >1 ventana={(w > 1).mean():.2%}  max ventanas={w.max()}")

## 6. Tipo de pregunta (primera palabra)

In [ ]:
QTYPES = ["what", "who", "how", "when", "which", "where", "why"]
def qtype(q):
    w = q.strip().lower().split()
    return w[0] if w and w[0] in QTYPES else "other"

qt = np.array([qtype(q) for q in train["question"]])
unans_arr = np.array(train_unans)
labels = QTYPES + ["other"]
qt_ans   = [int(((qt == t) & ~unans_arr).sum()) for t in labels]
qt_unans = [int(((qt == t) &  unans_arr).sum()) for t in labels]
for t, a, u in zip(labels, qt_ans, qt_unans):
    print(f"{t:<6} contestables={a:>6,}  no contestables={u:>6,}")

## 7. Gráficas → `eda_squad2.png`

In [ ]:
C_ANS, C_UNANS = "#2E7D32", "#C62828"
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# (a) Contestables vs no contestables por split
ax = axes[0, 0]
splits = ["train", "validation"]
ans_counts   = [len(train) - sum(train_unans), len(val) - sum(val_unans)]
unans_counts = [sum(train_unans), sum(val_unans)]
x = np.arange(2)
ax.bar(x - 0.2, ans_counts, 0.4, label="Contestables", color=C_ANS)
ax.bar(x + 0.2, unans_counts, 0.4, label="No contestables", color=C_UNANS)
for i in range(2):
    tot = ans_counts[i] + unans_counts[i]
    ax.text(x[i] - 0.2, ans_counts[i], f"{ans_counts[i]/tot:.1%}", ha="center", va="bottom", fontsize=9)
    ax.text(x[i] + 0.2, unans_counts[i], f"{unans_counts[i]/tot:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x, splits); ax.set_yscale("log"); ax.set_ylabel("Preguntas (escala log)")
ax.set_title("(a) Contestables vs. no contestables"); ax.legend()

# (b) Longitud del contexto en tokens de RoBERTa
ax = axes[0, 1]
ax.hist(ctx_tokens, bins=60, color="#1565C0", alpha=0.85)
ax.axvline(MAX_SEQ_LENGTH, color="black", ls="--", label=f"max_seq_length = {MAX_SEQ_LENGTH}")
ax.set_title("(b) Longitud de contexto (tokens RoBERTa, contextos únicos)")
ax.set_xlabel("Tokens"); ax.set_ylabel("Contextos"); ax.legend()

# (c) Ventanas por ejemplo
ax = axes[0, 2]
vals, cnts = np.unique(train_windows, return_counts=True)
ax.bar(vals.astype(str), cnts, color="#6A1B9A")
ax.set_yscale("log"); ax.set_title(f"(c) Ventanas por ejemplo (train, stride={DOC_STRIDE})")
ax.set_xlabel("Número de ventanas"); ax.set_ylabel("Ejemplos (escala log)")

# (d) Longitud de pregunta
ax = axes[1, 0]
bins = np.arange(0, 41)
ax.hist(q_words_ans, bins=bins, alpha=0.6, density=True, label="Contestables", color=C_ANS)
ax.hist(q_words_unans, bins=bins, alpha=0.6, density=True, label="No contestables", color=C_UNANS)
ax.set_title("(d) Longitud de pregunta (palabras, train)")
ax.set_xlabel("Palabras"); ax.set_ylabel("Proporción"); ax.legend()

# (e) Longitud de respuesta
ax = axes[1, 1]
ax.hist(np.clip(ans_words, 0, 30), bins=np.arange(1, 32), color="#EF6C00", alpha=0.85)
ax.set_title("(e) Longitud de respuesta (palabras, train; ≥30 agrupado)")
ax.set_xlabel("Palabras"); ax.set_ylabel("Respuestas")

# (f) Tipo de pregunta
ax = axes[1, 2]
x = np.arange(len(labels))
ax.bar(x - 0.2, qt_ans, 0.4, label="Contestables", color=C_ANS)
ax.bar(x + 0.2, qt_unans, 0.4, label="No contestables", color=C_UNANS)
ax.set_xticks(x, labels); ax.set_title("(f) Tipo de pregunta (primera palabra, train)")
ax.set_ylabel("Preguntas"); ax.legend()

fig.suptitle("SQuAD v2 — Análisis exploratorio (tokenizer: FacebookAI/roberta-base)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "eda_squad2.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Guardado:", os.path.join(LOGS_DIR, "eda_squad2.png"))

## 8. Estadísticas → `eda_stats.json`

In [ ]:
def pct(a, b): return round(100 * a / b, 2)

stats = {
    "dataset": "rajpurkar/squad_v2",
    "tokenizer": MODEL_CHECKPOINT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "doc_stride": DOC_STRIDE,
    "train_total": len(train),
    "val_total": len(val),
    "train_unanswerable": int(sum(train_unans)),
    "val_unanswerable": int(sum(val_unans)),
    "train_unanswerable_pct": pct(sum(train_unans), len(train)),
    "val_unanswerable_pct": pct(sum(val_unans), len(val)),
    "train_articles": len(set(train["title"])),
    "train_unique_contexts": len(unique_ctx),
    "context_len_chars_mean": round(float(ctx_chars.mean()), 1),
    "context_len_chars_max": int(ctx_chars.max()),
    "context_len_tokens_mean": round(float(ctx_tokens.mean()), 1),
    "context_len_tokens_median": float(np.median(ctx_tokens)),
    "context_len_tokens_p95": float(np.percentile(ctx_tokens, 95)),
    "context_len_tokens_max": int(ctx_tokens.max()),
    "contexts_longer_than_max_seq_pct": pct((ctx_tokens > MAX_SEQ_LENGTH).sum(), len(ctx_tokens)),
    "train_features": int(train_windows.sum()),
    "val_features": int(val_windows.sum()),
    "train_examples_multi_window_pct": pct((train_windows > 1).sum(), len(train_windows)),
    "val_examples_multi_window_pct": pct((val_windows > 1).sum(), len(val_windows)),
    "question_len_words_mean": round(float(q_words.mean()), 2),
    "answer_len_words_mean": round(float(ans_words.mean()), 2),
    "answer_len_words_median": float(np.median(ans_words)),
    "answer_len_words_max": int(ans_words.max()),
    "answer_len_chars_mean": round(float(ans_chars.mean()), 2),
    "answer_len_chars_max": int(ans_chars.max()),
    "question_types_train": {t: {"answerable": a, "unanswerable": u}
                             for t, a, u in zip(labels, qt_ans, qt_unans)},
}
with open(os.path.join(LOGS_DIR, "eda_stats.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(json.dumps(stats, indent=2, ensure_ascii=False))
print("✅ Guardado:", os.path.join(LOGS_DIR, "eda_stats.json"))

## Conclusiones para el entrenamiento

Revisa los números de arriba y anota aquí lo que observes. Puntos a mirar:
- ¿Las clases contestable / no contestable están balanceadas en validation? ¿Y en train?
- ¿Qué porcentaje de ejemplos necesita más de una ventana con `max_seq_length=384`?
  Eso justifica usar `doc_stride=128`.
- La mayoría de respuestas son cortas; `max_answer_length=30` tokens en el postproceso
  debería cubrirlas casi todas.

**Siguiente paso:** `02_train_roberta.ipynb`.

---
# Parte B · Training


**Modelo:** `FacebookAI/roberta-base` (checkpoint pretrained original, **sin** fine-tuning previo de QA).
**Tarea:** Extractive Question Answering con preguntas no contestables.

Etapas:
1. Configuración y detección de GPU.
2. Tokenización con *sliding window* (`max_seq_length=384`, `doc_stride=128`).
3. Fine-tuning con `Trainer` (3 epochs, LR 3e-5, scheduler lineal, warmup 10%).
4. Guardado del modelo final + curva de pérdida.
5. Evaluación en validation: EM y F1 global, contestables y no contestables.
6. Latencia de inferencia en **CPU**.

**Antes de empezar:** en Colab, `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4 o mejor)`.

**Si Colab se desconecta:** los checkpoints se guardan en Drive cada 1,000 pasos.
Vuelve a ejecutar el notebook desde el inicio y el entrenamiento **continúa desde el último checkpoint**.
Para empezar un experimento nuevo desde cero, borra la carpeta `checkpoints/` en Drive.

Archivos generados en `logs_roberta/`: `loss.png`, `train_results.json`, `eval_results.json`.
Fuera de `logs_roberta/` (solo en Drive): el modelo, los checkpoints y `predictions_val.json`
(lo usa el notebook 03).

## 1. Configuración

In [ ]:
import json, time, glob, platform, collections
import numpy as np
import torch
import matplotlib.pyplot as plt
import transformers
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments,
                          Trainer, TrainerCallback, default_data_collator, set_seed, pipeline)
from transformers.trainer_utils import get_last_checkpoint
import evaluate

CONFIG = {
    "model_checkpoint": "FacebookAI/roberta-base",
    "dataset": "rajpurkar/squad_v2",
    "max_train_samples": None,     # None = train completo. Ej.: 30000 para igualar el subset de ModernBERT
    "num_train_epochs": 3,
    "learning_rate": 3e-5,
    "lr_scheduler_type": "linear",
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_seq_length": 384,
    "doc_stride": 128,
    "effective_batch_size": 32,
    "per_device_train_batch_size": None,   # None = automático según la GPU. Si hay OOM, pon 8.
    "n_best": 20,
    "max_answer_length": 30,
    "null_threshold": 0.0,         # span si score(span) > score(<s>) + threshold
    "seed": 42,
}
set_seed(CONFIG["seed"])

## 2. GPU y tamaño de batch

In [ ]:
assert torch.cuda.is_available(), "No hay GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU."

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
use_bf16   = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere o superior (A100, L4)
use_fp16   = not use_bf16                                  # T4 / V100

bs = CONFIG["per_device_train_batch_size"] or (32 if gpu_mem_gb >= 35 else 16)
assert CONFIG["effective_batch_size"] % bs == 0
grad_accum = CONFIG["effective_batch_size"] // bs
CONFIG["per_device_train_batch_size"] = bs
CONFIG["gradient_accumulation_steps"] = grad_accum
CONFIG["precision"] = "bf16" if use_bf16 else "fp16"

print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
print(f"Batch por dispositivo: {bs} × acumulación {grad_accum} = efectivo {bs * grad_accum}")
print(f"Precisión mixta: {CONFIG['precision']}")

## 3. Datos y tokenización

- **Train:** cada ejemplo se divide en ventanas de 384 tokens. Si la respuesta no cae
  dentro de una ventana (o la pregunta no tiene respuesta), la etiqueta apunta al token
  `<s>` (posición 0), que funciona como "sin respuesta".
- **Validation (pérdida):** misma tokenización que train, para calcular `eval_loss`.
- **Validation (predicciones):** guardamos `offset_mapping` para convertir posiciones
  de tokens en texto del contexto.

In [ ]:
raw = load_dataset(CONFIG["dataset"])
raw_train, raw_val = raw["train"], raw["validation"]
if CONFIG["max_train_samples"]:
    raw_train = raw_train.shuffle(seed=CONFIG["seed"]).select(range(CONFIG["max_train_samples"]))
print(f"Train: {len(raw_train):,} ejemplos | Validation: {len(raw_val):,} ejemplos")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_checkpoint"])
MAX_LEN, STRIDE = CONFIG["max_seq_length"], CONFIG["doc_stride"]

In [ ]:
def tokenize(examples):
    return tokenizer(
        [q.strip() for q in examples["question"]], examples["context"],
        max_length=MAX_LEN, truncation="only_second", stride=STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )

def prepare_train_features(examples):
    enc = tokenize(examples)
    sample_map = enc.pop("overflow_to_sample_mapping")
    offsets_all = enc.pop("offset_mapping")
    starts, ends = [], []
    for i, offsets in enumerate(offsets_all):
        answer = examples["answers"][sample_map[i]]
        cls_index = enc["input_ids"][i].index(tokenizer.cls_token_id)   # 0 en RoBERTa
        seq_ids = enc.sequence_ids(i)
        if len(answer["answer_start"]) == 0:                           # no contestable
            starts.append(cls_index); ends.append(cls_index); continue
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        ctx_start = seq_ids.index(1)
        ctx_end = len(seq_ids) - 1 - seq_ids[::-1].index(1)
        if offsets[ctx_start][0] > start_char or offsets[ctx_end][1] < end_char:
            starts.append(cls_index); ends.append(cls_index); continue  # respuesta fuera de la ventana
        s = ctx_start
        while s <= ctx_end and offsets[s][0] <= start_char: s += 1
        e = ctx_end
        while e >= ctx_start and offsets[e][1] >= end_char: e -= 1
        starts.append(s - 1); ends.append(e + 1)
    enc["start_positions"], enc["end_positions"] = starts, ends
    return enc

def prepare_val_features(examples):
    enc = tokenize(examples)
    sample_map = enc.pop("overflow_to_sample_mapping")
    enc["example_id"] = []
    for i in range(len(enc["input_ids"])):
        enc["example_id"].append(examples["id"][sample_map[i]])
        seq_ids = enc.sequence_ids(i)
        enc["offset_mapping"][i] = [o if seq_ids[k] == 1 else None
                                    for k, o in enumerate(enc["offset_mapping"][i])]
    return enc

train_ds = raw_train.map(prepare_train_features, batched=True, remove_columns=raw_train.column_names,
                         desc="Tokenizando train")
val_loss_ds = raw_val.map(prepare_train_features, batched=True, remove_columns=raw_val.column_names,
                          desc="Tokenizando validation (loss)")
val_pred_ds = raw_val.map(prepare_val_features, batched=True, remove_columns=raw_val.column_names,
                          desc="Tokenizando validation (predicciones)")
print(f"Features — train: {len(train_ds):,} | val loss: {len(val_loss_ds):,} | val pred: {len(val_pred_ds):,}")

In [ ]:
# Comprobación rápida: la etiqueta de un ejemplo contestable debe reconstruir la respuesta
i = next(k for k in range(len(train_ds)) if train_ds[k]["start_positions"] != 0)
f = train_ds[i]
print("Span reconstruido:", repr(tokenizer.decode(f["input_ids"][f["start_positions"]:f["end_positions"] + 1])))

## 4. Modelo

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(CONFIG["model_checkpoint"])
num_params = model.num_parameters()
print(f"Parámetros: {num_params:,}")

## 5. Entrenamiento

`SessionTimer` registra en Drive cuánto dura cada sesión de entrenamiento, para poder
sumar el tiempo real aunque Colab se desconecte y haya que reanudar.

In [ ]:
SESSIONS_FILE = os.path.join(CKPT_DIR, "training_sessions.json")
os.makedirs(CKPT_DIR, exist_ok=True)
last_ckpt = get_last_checkpoint(CKPT_DIR)
if last_ckpt is None and os.path.exists(SESSIONS_FILE):
    os.remove(SESSIONS_FILE)        # entrenamiento nuevo: no arrastrar sesiones anteriores
print("Reanudando desde:" if last_ckpt else "Entrenamiento nuevo.", last_ckpt or "")

class SessionTimer(TrainerCallback):
    def _write(self, state):
        self.sessions[-1]["seconds"] = round(time.time() - self.t0, 1)
        self.sessions[-1]["last_step"] = state.global_step
        with open(SESSIONS_FILE, "w") as f:
            json.dump(self.sessions, f, indent=2)
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        self.sessions = json.load(open(SESSIONS_FILE)) if os.path.exists(SESSIONS_FILE) else []
        self.sessions.append({"gpu": gpu_name, "resumed_from": last_ckpt, "seconds": 0})
        self._write(state)
    def on_log(self, args, state, control, **kw):  self._write(state)
    def on_save(self, args, state, control, **kw): self._write(state)
    def on_train_end(self, args, state, control, **kw): self._write(state)

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=CONFIG["num_train_epochs"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=grad_accum,
    fp16=use_fp16, bf16=use_bf16,
    eval_strategy="epoch",
    save_strategy="steps", save_steps=1000, save_total_limit=2,
    logging_steps=100,
    seed=CONFIG["seed"],
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_loss_ds,
    data_collator=default_data_collator, processing_class=tokenizer,
    callbacks=[SessionTimer()],
)

In [ ]:
train_result = trainer.train(resume_from_checkpoint=last_ckpt)

## 6. Guardar modelo final y curva de pérdida

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

weights = glob.glob(os.path.join(MODEL_DIR, "*.safetensors")) + glob.glob(os.path.join(MODEL_DIR, "*.bin"))
weights_bytes = sum(os.path.getsize(p) for p in weights)
dir_bytes = sum(os.path.getsize(os.path.join(MODEL_DIR, f)) for f in os.listdir(MODEL_DIR))
print(f"Modelo guardado en {MODEL_DIR}")
print(f"Pesos: {weights_bytes/1024**2:.1f} MB | Carpeta completa: {dir_bytes/1024**2:.1f} MB")

sessions = json.load(open(SESSIONS_FILE))
train_seconds = sum(s["seconds"] for s in sessions)
print(f"Tiempo total de entrenamiento: {train_seconds/3600:.2f} h en {len(sessions)} sesión(es)")

In [ ]:
hist = trainer.state.log_history          # incluye lo registrado antes de reanudar
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h and "eval_loss" not in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(11, 5))
plt.plot(*zip(*tr), label="Train loss", color="#1565C0", alpha=0.85)
if ev:
    plt.plot(*zip(*ev), "o--", label="Validation loss (por epoch)", color="#C62828", lw=2)
steps_per_epoch = trainer.state.max_steps / CONFIG["num_train_epochs"]
for e in range(1, CONFIG["num_train_epochs"]):
    plt.axvline(e * steps_per_epoch, color="gray", ls=":", lw=1)
plt.xlabel("Paso de optimización"); plt.ylabel("Loss")
plt.title(f"RoBERTa-base / SQuAD v2 — LR={CONFIG['learning_rate']}, epochs={CONFIG['num_train_epochs']}, "
          f"batch efectivo={bs*grad_accum}")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "loss.png"), dpi=150)
plt.show()

## 7. Evaluación en validation (11,873 preguntas)

Para cada pregunta se combinan todas sus ventanas:
- **Mejor span:** máximo de `start_logit + end_logit` entre los `n_best` candidatos válidos.
- **Score de "sin respuesta":** `start_logit[<s>] + end_logit[<s>]` (mínimo entre ventanas).
- Se predice el span si `score_span > score_nulo + threshold`; si no, la respuesta es `""`.

La métrica oficial `squad_v2` cuenta como **correcta** una predicción vacía `""` en una
pregunta no contestable (EM = F1 = 1 para ese ejemplo).

In [ ]:
def postprocess(raw_examples, features, start_logits, end_logits, n_best=20, max_answer_length=30):
    feats_per_example = collections.defaultdict(list)
    for i, ex_id in enumerate(features["example_id"]):
        feats_per_example[ex_id].append(i)
    offsets_all = list(features["offset_mapping"])
    results = []
    for ex in raw_examples:
        best_text, best_score, min_null = "", -1e9, None
        for fi in feats_per_example[ex["id"]]:
            sl, el, offsets = start_logits[fi], end_logits[fi], offsets_all[fi]
            null_score = float(sl[0] + el[0])            # posición 0 = <s>
            min_null = null_score if min_null is None else min(min_null, null_score)
            for s in np.argsort(sl)[-1:-n_best - 1:-1]:
                for e in np.argsort(el)[-1:-n_best - 1:-1]:
                    if offsets[s] is None or offsets[e] is None: continue
                    if e < s or e - s + 1 > max_answer_length: continue
                    score = float(sl[s] + el[e])
                    if score > best_score:
                        best_score, best_text = score, ex["context"][offsets[s][0]:offsets[e][1]]
        results.append({"id": ex["id"], "best_span_text": best_text, "best_span_score": best_score,
                        "null_score": min_null, "score_diff": best_score - min_null})
    return results

def apply_threshold(results, threshold):
    return {r["id"]: (r["best_span_text"] if r["score_diff"] > threshold else "") for r in results}

squad_metric = evaluate.load("squad_v2")
def squad_v2_scores(pred_dict, raw_examples):
    preds = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in pred_dict.items()]
    refs = [{"id": ex["id"], "answers": ex["answers"]} for ex in raw_examples]
    return squad_metric.compute(predictions=preds, references=refs)

In [ ]:
eval_loss_metrics = trainer.evaluate()          # eval_loss del modelo final

t0 = time.time()
pred_out = trainer.predict(val_pred_ds.remove_columns(["example_id", "offset_mapping"]))
start_logits, end_logits = pred_out.predictions
results = postprocess(raw_val, val_pred_ds, start_logits, end_logits,
                      CONFIG["n_best"], CONFIG["max_answer_length"])
gpu_eval_seconds = time.time() - t0

scores = squad_v2_scores(apply_threshold(results, CONFIG["null_threshold"]), raw_val)
print(f"EM global {scores['exact']:.2f} | F1 global {scores['f1']:.2f}")
print(f"Contestables    EM {scores['HasAns_exact']:.2f} | F1 {scores['HasAns_f1']:.2f}  (n={scores['HasAns_total']})")
print(f"No contestables EM {scores['NoAns_exact']:.2f} | F1 {scores['NoAns_f1']:.2f}  (n={scores['NoAns_total']})")

with open(PRED_FILE, "w", encoding="utf-8") as f:     # lo usa el notebook 03
    json.dump(results, f, ensure_ascii=False)

## 8. Latencia de inferencia en CPU

Se carga el modelo guardado **en CPU** y se mide el tiempo de una pregunta completa
(tokenización + ventanas + modelo + postproceso) con el `pipeline` de Transformers.

⚠️ La CPU de Colab no es la misma que la de tu laptop. Para el reporte final conviene
repetir esta medición en la máquina donde corra la API.

In [ ]:
cpu_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_DIR)   # se carga en CPU
qa_cpu = pipeline("question-answering", model=cpu_model, tokenizer=tokenizer, device=-1)
qa_kwargs = dict(max_seq_len=MAX_LEN, doc_stride=STRIDE, max_answer_len=CONFIG["max_answer_length"],
                 handle_impossible_answer=True)

def time_query(ex):
    t = time.perf_counter()
    qa_cpu(question=ex["question"], context=ex["context"], **qa_kwargs)
    return (time.perf_counter() - t) * 1000

sample = raw_val.shuffle(seed=CONFIG["seed"]).select(range(33))
for ex in sample.select(range(3)): time_query(ex)          # calentamiento
lat = np.array([time_query(ex) for ex in sample.select(range(3, 33))])

longest = raw_val[int(np.argmax([len(c) for c in raw_val["context"]]))]
lat_long = np.array([time_query(longest) for _ in range(5)])

cpu_latency = {
    "device": "cpu",
    "cpu": platform.processor() or platform.machine(),
    "cpu_count": os.cpu_count(),
    "torch_threads": torch.get_num_threads(),
    "environment": "Google Colab" if IN_COLAB else "local",
    "method": "transformers pipeline, 30 preguntas aleatorias de validation (3 de calentamiento)",
    "mean_ms": round(float(lat.mean()), 1),
    "p50_ms": round(float(np.percentile(lat, 50)), 1),
    "p95_ms": round(float(np.percentile(lat, 95)), 1),
    "long_context_chars": len(longest["context"]),
    "long_context_mean_ms": round(float(lat_long.mean()), 1),
}
print(json.dumps(cpu_latency, indent=2))

## 9. Guardar `train_results.json` y `eval_results.json`

In [ ]:
train_results = {
    "model_checkpoint": CONFIG["model_checkpoint"],
    "config": CONFIG,
    "data": {
        "train_examples": len(raw_train), "train_features": len(train_ds),
        "val_examples": len(raw_val), "val_features": len(val_pred_ds),
    },
    "hardware": {"gpu": gpu_name, "gpu_memory_gb": round(gpu_mem_gb, 1), "precision": CONFIG["precision"]},
    "num_parameters": num_params,
    "model_weights_mb": round(weights_bytes / 1024**2, 1),
    "model_dir_mb": round(dir_bytes / 1024**2, 1),
    "training_time": {
        "total_seconds": round(train_seconds, 1),
        "total_hours": round(train_seconds / 3600, 2),
        "num_sessions": len(sessions),
        "sessions": sessions,
        "note": "Suma de todas las sesiones (incluye la evaluación de loss al final de cada epoch).",
    },
    "global_steps": trainer.state.global_step,
    "final_train_loss_logged": tr[-1][1] if tr else None,
    "trainer_metrics_last_session": train_result.metrics,
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
}

eval_results = {
    "split": "validation",
    "num_examples": len(raw_val),
    "null_threshold": CONFIG["null_threshold"],
    "overall":      {"exact_match": scores["exact"],        "f1": scores["f1"],        "count": scores["total"]},
    "answerable":   {"exact_match": scores["HasAns_exact"], "f1": scores["HasAns_f1"], "count": scores["HasAns_total"]},
    "unanswerable": {"exact_match": scores["NoAns_exact"],  "f1": scores["NoAns_f1"],  "count": scores["NoAns_total"]},
    "eval_loss": eval_loss_metrics.get("eval_loss"),
    "gpu_eval_seconds": round(gpu_eval_seconds, 1),
    "cpu_latency": cpu_latency,
    "metric": "evaluate.load('squad_v2'); '' en una pregunta no contestable cuenta como correcto",
}

for name, obj in [("train_results.json", train_results), ("eval_results.json", eval_results)]:
    with open(os.path.join(LOGS_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    print("✅ Guardado:", os.path.join(LOGS_DIR, name))

In [ ]:
print("=" * 60)
print("RESUMEN — RoBERTa-base / SQuAD v2")
print("=" * 60)
print(f"Train: {len(raw_train):,} ejemplos ({len(train_ds):,} features) | Val: {len(raw_val):,}")
print(f"Epochs {CONFIG['num_train_epochs']} | LR {CONFIG['learning_rate']} {CONFIG['lr_scheduler_type']} "
      f"warmup {CONFIG['warmup_ratio']:.0%} | batch {bs}×{grad_accum}={bs*grad_accum}")
print(f"GPU: {gpu_name} ({CONFIG['precision']}) | Tiempo: {train_seconds/3600:.2f} h")
print(f"Parámetros: {num_params:,} | Pesos: {weights_bytes/1024**2:.0f} MB")
print(f"EM {scores['exact']:.2f} / F1 {scores['f1']:.2f} "
      f"(HasAns {scores['HasAns_f1']:.2f} F1, NoAns {scores['NoAns_f1']:.2f} F1)")
print(f"Latencia CPU media: {cpu_latency['mean_ms']:.0f} ms")

---
# Parte C · Error Analysis


Usa las predicciones de validation guardadas por `02_train_roberta.ipynb`
(`predictions_val.json` en Drive), así que **no necesita GPU ni volver a correr el modelo**.

Etapas:
1. **Búsqueda de threshold** de "sin respuesta" → `threshold_search.png`.
2. **Clasificación de errores** con la normalización oficial de SQuAD.
3. Ejemplos de predicciones **correctas e incorrectas**, para contestables y no contestables.
4. Errores por tipo de pregunta y por longitud del contexto.
5. Guardado de `error_analysis.json`.
6. Generación automática de `README.md` y `ROBERTA_COMPARISON.md` a partir de los JSON reales.

## 1. Cargar predicciones y datos

In [ ]:
import json, re, string, collections, random
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import evaluate

assert os.path.exists(PRED_FILE), f"No existe {PRED_FILE}. Ejecuta primero 02_train_roberta.ipynb."
results = json.load(open(PRED_FILE, encoding="utf-8"))
eval_results = json.load(open(os.path.join(LOGS_DIR, "eval_results.json"), encoding="utf-8"))
REPORTED_THRESHOLD = eval_results["null_threshold"]

raw_val = load_dataset("rajpurkar/squad_v2", split="validation")
by_id = {ex["id"]: ex for ex in raw_val}
assert len(results) == len(raw_val)
print(f"{len(results):,} predicciones cargadas | threshold usado en eval_results.json: {REPORTED_THRESHOLD}")

## 2. Búsqueda de threshold

`score_diff = score(mejor span) − score(<s>)`. Se responde con el span si
`score_diff > threshold`; si no, se devuelve `""`.

- Threshold **bajo** → responde casi siempre: sube F1 en contestables, baja en no contestables.
- Threshold **alto** → se abstiene más: lo contrario.

⚠️ El threshold se elige sobre el mismo validation set que se reporta, así que el
"mejor F1" de esta búsqueda es **optimista**. Las métricas oficiales del modelo son las
de `eval_results.json` (threshold fijo = 0.0). El valor encontrado aquí sirve para
configurar `NULL_THRESHOLD` en la API.

In [ ]:
squad_metric = evaluate.load("squad_v2")
refs = [{"id": ex["id"], "answers": ex["answers"]} for ex in raw_val]

def apply_threshold(threshold):
    return {r["id"]: (r["best_span_text"] if r["score_diff"] > threshold else "") for r in results}

def scores_at(threshold):
    preds = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0}
             for k, v in apply_threshold(threshold).items()]
    return squad_metric.compute(predictions=preds, references=refs)

thresholds = np.round(np.arange(-5.0, 5.01, 0.5), 2).tolist()
search = []
for t in thresholds:
    s = scores_at(t)
    search.append({"threshold": t, "exact": s["exact"], "f1": s["f1"],
                   "HasAns_f1": s["HasAns_f1"], "NoAns_f1": s["NoAns_f1"]})
    print(f"t={t:+.1f}  EM={s['exact']:.2f}  F1={s['f1']:.2f}  HasAns F1={s['HasAns_f1']:.2f}  NoAns F1={s['NoAns_f1']:.2f}")

best = max(search, key=lambda r: r["f1"])
print(f"\nMejor threshold por F1 global: {best['threshold']:+.1f} → F1 {best['f1']:.2f}")

In [ ]:
plt.figure(figsize=(11, 5))
T = [r["threshold"] for r in search]
plt.plot(T, [r["f1"] for r in search], "o-", label="F1 global", color="#1565C0", lw=2)
plt.plot(T, [r["exact"] for r in search], "s-", label="EM global", color="#6A1B9A")
plt.plot(T, [r["HasAns_f1"] for r in search], "--", label="F1 contestables", color="#2E7D32")
plt.plot(T, [r["NoAns_f1"] for r in search], "--", label="F1 no contestables", color="#C62828")
plt.axvline(best["threshold"], color="black", ls=":", label=f"Mejor threshold = {best['threshold']:+.1f}")
plt.axvline(REPORTED_THRESHOLD, color="gray", ls="-", alpha=0.4, label=f"Threshold reportado = {REPORTED_THRESHOLD}")
plt.xlabel("Threshold (score_span − score_nulo)"); plt.ylabel("Score (%)")
plt.title("RoBERTa-base / SQuAD v2 validation — métricas vs. threshold de no-respuesta")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "threshold_search.png"), dpi=150)
plt.show()

## 3. Clasificación de errores

Se usa la **normalización oficial de SQuAD** (minúsculas, sin puntuación, sin artículos
*a/an/the*, espacios colapsados), así que `"the Normans"` y `"Normans"` cuentan igual.

Categorías (con el threshold reportado, para que coincidan con `eval_results.json`):

| Tipo de pregunta | Categoría | Significado |
|---|---|---|
| contestable | `correct` | EM = 1 |
| contestable | `partial_overlap` | span con solapamiento parcial (0 < F1 < 1) |
| contestable | `wrong_span` | span sin ninguna palabra en común (F1 = 0) |
| contestable | `false_negative` | el modelo dijo "sin respuesta" y sí había |
| no contestable | `correct` | el modelo devolvió `""` |
| no contestable | `false_positive` | el modelo inventó un span |

In [ ]:
ANALYSIS_THRESHOLD = REPORTED_THRESHOLD

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def token_f1(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not p or not g:
        return float(p == g)
    same = sum((collections.Counter(p) & collections.Counter(g)).values())
    if same == 0:
        return 0.0
    prec, rec = same / len(p), same / len(g)
    return 2 * prec * rec / (prec + rec)

QTYPES = ["what", "who", "how", "when", "which", "where", "why"]
def qtype(q):
    w = q.strip().lower().split()
    return w[0] if w and w[0] in QTYPES else "other"

preds = apply_threshold(ANALYSIS_THRESHOLD)
rows = []
for r in results:
    ex = by_id[r["id"]]
    pred = preds[r["id"]]
    golds = ex["answers"]["text"]
    answerable = len(golds) > 0
    gold_list = golds if answerable else [""]
    em = max(float(normalize_answer(pred) == normalize_answer(g)) for g in gold_list)
    f1 = max(token_f1(pred, g) for g in gold_list)
    if em == 1:
        cat = "correct"
    elif not answerable:
        cat = "false_positive"
    elif pred == "":
        cat = "false_negative"
    elif f1 > 0:
        cat = "partial_overlap"
    else:
        cat = "wrong_span"
    rows.append({"id": r["id"], "question": ex["question"], "context": ex["context"],
                 "gold": golds, "prediction": pred, "answerable": answerable,
                 "category": cat, "em": em, "f1": round(f1, 3),
                 "score_diff": round(r["score_diff"], 3), "qtype": qtype(ex["question"])})

# Verificación: los promedios deben coincidir con la métrica oficial de eval_results.json
em_all, f1_all = 100 * np.mean([x["em"] for x in rows]), 100 * np.mean([x["f1"] for x in rows])
print(f"EM {em_all:.2f} / F1 {f1_all:.2f}  (eval_results.json: "
      f"{eval_results['overall']['exact_match']:.2f} / {eval_results['overall']['f1']:.2f})")

In [ ]:
def breakdown(subset):
    c = collections.Counter(x["category"] for x in subset)
    return {k: c[k] for k in sorted(c)}

answerable_rows   = [x for x in rows if x["answerable"]]
unanswerable_rows = [x for x in rows if not x["answerable"]]
errors = [x for x in rows if x["category"] != "correct"]

print(f"Errores totales: {len(errors):,} de {len(rows):,} ({len(errors)/len(rows):.1%})\n")
for name, subset in [("Contestables", answerable_rows), ("No contestables", unanswerable_rows)]:
    print(f"{name} (n={len(subset):,})")
    for k, v in breakdown(subset).items():
        print(f"   {k:<16} {v:>6,}  ({v/len(subset):.1%})")

## 4. Errores por tipo de pregunta y por longitud de contexto

In [ ]:
def error_rate_by(key_fn, subset):
    groups = collections.defaultdict(list)
    for x in subset:
        groups[key_fn(x)].append(x["category"] != "correct")
    return {k: {"n": len(v), "error_rate_pct": round(100 * np.mean(v), 2)} for k, v in groups.items()}

by_qtype = error_rate_by(lambda x: x["qtype"], rows)
by_qtype = dict(sorted(by_qtype.items(), key=lambda kv: -kv[1]["n"]))

ctx_len = np.array([len(x["context"]) for x in rows])
edges = np.percentile(ctx_len, [25, 50, 75]).astype(int)
def ctx_bin(x):
    n = len(x["context"])
    if n <= edges[0]: return f"Q1 (≤{edges[0]} chars)"
    if n <= edges[1]: return f"Q2 (≤{edges[1]})"
    if n <= edges[2]: return f"Q3 (≤{edges[2]})"
    return f"Q4 (>{edges[2]})"
by_ctx = dict(sorted(error_rate_by(ctx_bin, rows).items()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(by_qtype.keys(), [v["error_rate_pct"] for v in by_qtype.values()], color="#1565C0")
axes[0].set_title("Tasa de error por tipo de pregunta"); axes[0].set_ylabel("% errores")
axes[1].bar(by_ctx.keys(), [v["error_rate_pct"] for v in by_ctx.values()], color="#EF6C00")
axes[1].set_title("Tasa de error por cuartil de longitud de contexto"); axes[1].set_ylabel("% errores")
plt.tight_layout(); plt.show()   # solo se muestra en el notebook; los datos van a error_analysis.json
print(json.dumps({"by_question_type": by_qtype, "by_context_length": by_ctx}, indent=2, ensure_ascii=False))

## 5. Ejemplos de predicciones correctas e incorrectas

In [ ]:
random.seed(42)
def sample(cat, answerable, k=8):
    pool = [x for x in rows if x["category"] == cat and x["answerable"] == answerable]
    picked = random.sample(pool, min(k, len(pool)))
    return [{"id": x["id"], "question": x["question"], "context": x["context"][:400] + "...",
             "gold": x["gold"], "prediction": x["prediction"], "f1": x["f1"],
             "score_diff": x["score_diff"]} for x in picked]

examples = {
    "correct_answerable":   sample("correct", True),
    "correct_unanswerable": sample("correct", False),
    "partial_overlap":      sample("partial_overlap", True),
    "wrong_span":           sample("wrong_span", True),
    "false_negative":       sample("false_negative", True),
    "false_positive":       sample("false_positive", False),
}

for name, items in examples.items():
    print(f"\n=== {name} ({len(items)} ejemplos) ===")
    for x in items[:3]:
        print(f"Q: {x['question']}")
        print(f"   gold: {x['gold'] or '(sin respuesta)'} | pred: {x['prediction']!r} | score_diff={x['score_diff']}")

## 6. Guardar `error_analysis.json`

In [ ]:
error_analysis = {
    "analysis_threshold": ANALYSIS_THRESHOLD,
    "normalization": "oficial SQuAD (lower, sin puntuación, sin artículos, espacios colapsados)",
    "total_examples": len(rows),
    "total_errors": len(errors),
    "error_rate_pct": round(100 * len(errors) / len(rows), 2),
    "breakdown_answerable": breakdown(answerable_rows),
    "breakdown_unanswerable": breakdown(unanswerable_rows),
    "by_question_type": by_qtype,
    "by_context_length": by_ctx,
    "threshold_search": {
        "best_threshold": best["threshold"],
        "best_f1": best["f1"],
        "best_exact": best["exact"],
        "note": "Elegido sobre el mismo validation set: valor optimista; úsalo para NULL_THRESHOLD en la API.",
        "grid": search,
    },
    "examples": examples,
}
with open(os.path.join(LOGS_DIR, "error_analysis.json"), "w", encoding="utf-8") as f:
    json.dump(error_analysis, f, indent=2, ensure_ascii=False)
print("✅ Guardado:", os.path.join(LOGS_DIR, "error_analysis.json"))

## 7. Generar `README.md` y `ROBERTA_COMPARISON.md`

Se escriben **solo con los valores de los JSON** de `logs_roberta/`. Los números de
ModernBERT de la comparación vienen de `logs_modernBert/` en el repositorio del equipo.

In [ ]:
tr = json.load(open(os.path.join(LOGS_DIR, "train_results.json"), encoding="utf-8"))
ev = eval_results
eda = json.load(open(os.path.join(LOGS_DIR, "eda_stats.json"), encoding="utf-8"))
cfg, lat = tr["config"], ev["cpu_latency"]
f2 = lambda v: f"{v:.2f}"

readme = f"""# logs_roberta — RoBERTa-base fine-tuned en SQuAD v2

Resultados reales del fine-tuning de `{tr['model_checkpoint']}` para Extractive QA en SQuAD v2.
Generado automáticamente por `pipeline_roberta/03_error_analysis_roberta.ipynb`.

## Resultados (validation, {ev['num_examples']:,} preguntas, threshold = {ev['null_threshold']})

| Subconjunto | EM | F1 | n |
|---|---|---|---|
| **Global** | **{f2(ev['overall']['exact_match'])}** | **{f2(ev['overall']['f1'])}** | {ev['overall']['count']:,} |
| Contestables | {f2(ev['answerable']['exact_match'])} | {f2(ev['answerable']['f1'])} | {ev['answerable']['count']:,} |
| No contestables | {f2(ev['unanswerable']['exact_match'])} | {f2(ev['unanswerable']['f1'])} | {ev['unanswerable']['count']:,} |

Mejor threshold en la búsqueda: **{error_analysis['threshold_search']['best_threshold']:+.1f}**
(F1 {f2(error_analysis['threshold_search']['best_f1'])}; optimista porque se eligió sobre validation).

## Entrenamiento

| Parámetro | Valor |
|---|---|
| Modelo base | `{tr['model_checkpoint']}` |
| Train | {tr['data']['train_examples']:,} ejemplos ({tr['data']['train_features']:,} ventanas) |
| Validation | {tr['data']['val_examples']:,} ejemplos |
| Epochs | {cfg['num_train_epochs']} |
| Learning rate | {cfg['learning_rate']} ({cfg['lr_scheduler_type']}, warmup {cfg['warmup_ratio']:.0%}) |
| Batch | {cfg['per_device_train_batch_size']} × {cfg['gradient_accumulation_steps']} = {cfg['effective_batch_size']} efectivo |
| max_seq_length / doc_stride | {cfg['max_seq_length']} / {cfg['doc_stride']} |
| GPU | {tr['hardware']['gpu']} ({tr['hardware']['precision']}) |
| Tiempo de entrenamiento | {tr['training_time']['total_hours']} h ({tr['training_time']['num_sessions']} sesión(es)) |
| Parámetros | {tr['num_parameters']:,} |
| Tamaño de pesos | {tr['model_weights_mb']} MB |
| Latencia CPU (media / p95) | {lat['mean_ms']} / {lat['p95_ms']} ms ({lat['environment']}, {lat['cpu_count']} CPUs) |

## Archivos

| Archivo | Contenido |
|---|---|
| `eda_squad2.png`, `eda_stats.json` | Análisis exploratorio de SQuAD v2 |
| `loss.png` | Pérdida de entrenamiento y validación |
| `train_results.json` | Configuración exacta, hardware, tiempo, tamaño |
| `eval_results.json` | EM/F1 global, contestables y no contestables; latencia CPU |
| `threshold_search.png` | Métricas vs. threshold de no-respuesta |
| `error_analysis.json` | Tipos de error y ejemplos correctos/incorrectos |
| `ROBERTA_COMPARISON.md` | Ficha comparativa con ModernBERT |

![EDA](eda_squad2.png)
![Loss](loss.png)
![Threshold](threshold_search.png)
"""

MB = {"name": "ModernBERT-base (Celeste)", "train": "30,000", "epochs": 2, "em": 77.67, "f1": 81.04,
      "has_em": 72.39, "has_f1": 79.14, "no_em": 82.94, "no_f1": 82.94,
      "params": "149,606,402", "size": "572 MB", "time": "2.85 h", "gpu": "RTX 3050 Ti 4GB"}

comparison = f"""# Ficha comparativa — RoBERTa-base vs. ModernBERT-base

Valores de RoBERTa-base: `logs_roberta/*.json`. Valores de ModernBERT-base: `logs_modernBert/`
(documentados por Celeste). Ambos evaluados en el validation completo de SQuAD v2 con `evaluate.load("squad_v2")`.

| | RoBERTa-base | {MB['name']} |
|---|---|---|
| Checkpoint | `{tr['model_checkpoint']}` | `answerdotai/ModernBERT-base` |
| Ejemplos de train | {tr['data']['train_examples']:,} | {MB['train']} |
| Epochs | {cfg['num_train_epochs']} | {MB['epochs']} |
| LR / warmup | {cfg['learning_rate']} / {cfg['warmup_ratio']:.0%} | 3e-05 / 10% |
| Batch efectivo | {cfg['effective_batch_size']} | 32 |
| max_seq_length / stride | {cfg['max_seq_length']} / {cfg['doc_stride']} | 384 / 128 |
| GPU | {tr['hardware']['gpu']} | {MB['gpu']} |
| Tiempo de entrenamiento | {tr['training_time']['total_hours']} h | {MB['time']} |
| **EM global** | **{f2(ev['overall']['exact_match'])}** | **{MB['em']}** |
| **F1 global** | **{f2(ev['overall']['f1'])}** | **{MB['f1']}** |
| EM / F1 contestables | {f2(ev['answerable']['exact_match'])} / {f2(ev['answerable']['f1'])} | {MB['has_em']} / {MB['has_f1']} |
| EM / F1 no contestables | {f2(ev['unanswerable']['exact_match'])} / {f2(ev['unanswerable']['f1'])} | {MB['no_em']} / {MB['no_f1']} |
| Parámetros | {tr['num_parameters']:,} | {MB['params']} |
| Tamaño de pesos | {tr['model_weights_mb']} MB | {MB['size']} |
| Latencia CPU media | {lat['mean_ms']} ms ({lat['environment']}) | ~1,215–1,450 ms (laptop, vía API) |

**Cuidado al comparar:** los modelos no se entrenaron con la misma cantidad de datos ni epochs,
y la latencia se midió en máquinas y formas distintas (pipeline en {lat['environment']} vs. llamadas a la API en laptop).

## Errores de RoBERTa-base (threshold {error_analysis['analysis_threshold']})

- Contestables: {error_analysis['breakdown_answerable']}
- No contestables: {error_analysis['breakdown_unanswerable']}
"""

for name, text in [("README.md", readme), ("ROBERTA_COMPARISON.md", comparison)]:
    with open(os.path.join(LOGS_DIR, name), "w", encoding="utf-8") as f:
        f.write(text)
    print("✅ Guardado:", os.path.join(LOGS_DIR, name))

## 8. Descargar `logs_roberta/`

En Colab, la carpeta ya está en Drive (`MyDrive/roberta_squad2/logs_roberta/`).
Esta celda además crea un `.zip` y lo descarga a tu computadora.

In [ ]:
import shutil
print(sorted(os.listdir(LOGS_DIR)))
zip_path = shutil.make_archive(os.path.join(BASE_DIR, "logs_roberta"), "zip", BASE_DIR, "logs_roberta")
print("ZIP:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)